# AIR Homework 2 - Task and Motion Planning

## Assignment Overview

In the TAMP tutorial, we connected a task planner to a motion planner. The task planner plans, *then* the robot executes the plan one action at a time, with a skill and a motion plan for each action, and replans when it notices that something went wrong. In this assignment, your robot must carry out longer tasks in the N-Table Blocks World, in a world that is harder to work in: a wall with a window stands between the robot and one of the tables, plans fail during execution, an adversary moves blocks around, and some of the actions that the task planner chooses are impossible for the robot.

You will implement the following components:
1. **Execution Monitoring and Replanning**: Make the sampler of `put-down` aware of the robot's reach, compute the abstract state that the task planner expects after each action of a plan, and close the loop: execute a plan while comparing the world with these expectations, and replan as soon as they disagree.
2. **Integrated Task and Motion Planning**: Let information flow from the motion layer back to the task planner. Measure the cost of motions with the motion planner and write it into the PDDL model, and simulate plans at the motion level before executing them, so that the robot learns which actions are infeasible and tells the task planner about them.

To complete the assignment, read through the task notebooks and fill in the missing pieces of code in the `task*.py` files. You are responsible for testing and debugging your implementation.

## Submission Instructions

**FAILURE TO FOLLOW THESE INSTRUCTIONS MAY RESULT IN A PENALTY OR REJECTION OF YOUR SUBMISSION.**

You are required to submit a single zip file named HW2_\<YOUR_ID\>.zip, replacing \<YOUR_ID\> with your unique identifier. The zip file must contain the following files at its top level (not inside a folder):
- `task1.py` and `task2.py` with your code implementations.

Do not rename the task files.

⚠️ Warning: These will be the only files that will be examined - so any change you make to other files, including the notebooks, will not be considered.

All questions in the notebooks are purely for thought. You do not have to answer them.

Your code will undergo automated testing to ensure correctness and efficiency. In case of failure, you will receive feedback that is proportionate to the issues found. Please ensure your code is well-documented and follows best practices.

## Setup

This assignment runs locally in the course's conda environment. If you haven't done so yet, create the environment from `environment.yml`, activate it, and launch Jupyter:

```bash
conda env create -f environment.yml
conda activate air
jupyter lab
```

The assignment is split into the following files:
- `start_here.ipynb` - this notebook, with the assignment overview, the submission instructions, and our working environment.
- `task1.ipynb` and `task2.ipynb` - one notebook for each task, which walks you through the task and runs your code. Task 1 covers execution monitoring and replanning, and Task 2 covers integrated task and motion planning.
- `task1.py` and `task2.py` - one file for each task. All the code you need to write is in these files, marked with `#TODO` comments.
- `scene.py` - builds our simulation environment and provides the tools from the previous tutorials. This file is provided, do not change it.

Each task notebook imports these files as modules in its setup section (for example, `import task1 as t1`) and calls your code through them (for example, `t1.execute_with_monitoring(goal)`). Your changes are reloaded automatically, so after saving a file you only need to rerun the relevant cells.

The tasks build on each other, so complete them in order. Each task notebook starts with a setup section that loads our working environment and imports your code from the previous task. This way, you can run each notebook on its own.

Many cells of the task notebooks move the robot through a whole plan, and take a minute or two to run.

In [ ]:
%load_ext autoreload
%autoreload 2

# Set the EGL backend for MuJoCo on Linux (no effect on other operating systems)
import platform

if platform.system() == "Linux":
  %env MUJOCO_GL=egl

## Our Working Environment

Our environment is the N-Table Blocks World of the TAMP tutorial, rebuilt in plain MuJoCo. The specifications of the environment are as follows:
1. Three tables, `t1`, `t2` and `t3`, around a UR5e robot. The robot faces `t2`, with `t1` on its right and `t3` on its left.
2. A wall with a window (a serving hatch) stands between the robot and table `t1`. The robot can reach `t1` only through the window, which gives the motion planner something to do.
3. The robot has a suction cup instead of a gripper. When the suction is on, the cup holds the block it touches (we simulate this with a weld constraint), so picking up a block takes no finger motions.
4. The robot uses position actuation: the servos in its actuators track the target configuration we set, like the PD controller of the control tutorial.
5. Five blocks, 8 cm wide and 4 cm high, named `b1` to `b5`, in four colors. As in the PDDL problems we will write, `b2` starts on `b1` on table `t1`, `b3` is on `t2`, and `b4` and `b5` are on `t3`.

In [ ]:
# The scene is defined in scene.py. The first import downloads the UR5e robot MJCF files
# from DeepMind's MuJoCo Menagerie
import scene

import mediapy as media

print(f"UR5e robot MJCF path: {scene.UR5E_XML}\n")
print("blocks:", {block: color for block, color in scene.BLOCK_COLORS.items()})

scene.load_scene()
media.show_images([scene.render("overview"), scene.render("front"), scene.render("top")], titles=["overview", "front", "top"])

`scene.py` also provides the tools of the previous tutorials, which the task files use:

| what | functions |
|---|---|
| the world | `load_scene(layout)`, `reset()`, `render(camera)`, `show_video(frames)`, `set_seed(seed)`, `get_block_positions()` |
| moving the robot | `HOME`, `get_configuration()`, `set_configuration(q)` (teleport), `move_to(q)` (in a straight line), `follow_path(waypoints)`, `go_to(q)` (plan and follow) |
| inverse kinematics | `analytical_ik(position, rotation)`, `FACING_DOWN`, `choose_solution(solutions, reference)` |
| collisions | `in_collision(q)`, `robot_in_collision()` |
| the suction cup | `grasp()`, `release()`, `grasped_block()` |
| motion planning | `plan(q_start, q_goal)`: an OMPL path from one configuration to another, or `None` |

`load_scene(layout)` builds the world with any set of blocks, given as a dictionary from block names to colors and positions, and `load_scene()` gives the default scene above. Since it builds a new model, it replaces `scene.model`, `scene.data` and `scene.BLOCKS`. Always read them through the module, as `scene.data`, and never with `from scene import data`, which keeps the old ones.

Let's plan a motion to above block `b3` and follow it. Every time you run the cell, the motion planner finds a different path.

In [ ]:
position, rotation = scene.pose_above("b3")
q_goal = scene.choose_solution(scene.analytical_ik(position, rotation), scene.get_configuration())

scene.reset()
success, frames = scene.go_to(q_goal)
print("reached the goal:", success)
scene.show_video(frames)

**Next:** continue to Task 1 in `task1.ipynb`.